In [30]:
!pip install rapidfuzz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\dacdatalabs.ojt7\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


# Imports and Data

In [31]:
import os
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz, process

addr = pd.read_excel("../../data/sample_address.xlsx")
dim = pd.read_csv("../../data/mapping/dim_location_20260415_v3.csv",encoding="latin1")
aliases = pd.read_csv("../../data/utils/ph_address_alias_extended_v4.csv",encoding="latin1")

In [32]:
addr.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 1 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   order_deliveraddress  1000 non-null   str  
dtypes: str(1)
memory usage: 41.5 KB


In [33]:
dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 42011 entries, 0 to 42010
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   psgc_10_digit  42011 non-null  int64
 1   region_name    42010 non-null  str  
 2   province_name  42011 non-null  str  
 3   city_name      42011 non-null  str  
 4   barangay_name  42010 non-null  str  
 5   region_code    42011 non-null  int64
 6   province_code  42011 non-null  int64
 7   city_code      42011 non-null  int64
 8   barangay_code  42011 non-null  int64
dtypes: int64(5), str(4)
memory usage: 5.1 MB


In [34]:
aliases.info()


<class 'pandas.DataFrame'>
RangeIndex: 523 entries, 0 to 522
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   pattern      523 non-null    str  
 1   replacement  523 non-null    str  
 2   Unnamed: 2   1 non-null      str  
dtypes: str(3)
memory usage: 22.0 KB


In [35]:
aliases.drop("Unnamed: 2", axis=1, inplace=True)

# Preprocessing

In [36]:
addr["raw_address"] = addr["order_deliveraddress"]
addr.head()

,order_deliveraddress,raw_address
0,Santo Domingo St,Santo Domingo St
1,Tagumpay sindalan CSFP,Tagumpay sindalan CSFP
2,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL..."
3,Cainta Rizal,Cainta Rizal
4,2079 a elias st sta cruz,2079 a elias st sta cruz


## Cleaning

In [37]:
def clean_text(x):
    return str(x).strip().upper()

for col in ["region_name", "province_name", "city_name", "barangay_name"]:
    dim[col] = dim[col].apply(clean_text)

In [38]:
#creating aliases dictionary
alias_dict = dict(zip(aliases["pattern"], aliases["replacement"]))

## Normalization

In [39]:
import re

def normalize(text):
    text = str(text).upper()

    # remove punctuation
    text = re.sub(r"[^A-Z0-9\s]", " ", text)

    # apply aliases
    for pattern, replacement in alias_dict.items():
        text = re.sub(rf"\b{pattern}\b", replacement, text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [40]:
addr["clean_address"] = addr["raw_address"].apply(normalize)

# Precompute Lookup Maps

In [41]:
from collections import defaultdict

city_to_province = defaultdict(set)
barangay_to_city = defaultdict(set)
province_to_region = defaultdict(set)

for _, row in dim.iterrows():
    city_to_province[row["city_name"]].add(row["province_name"])
    barangay_to_city[row["barangay_name"]].add(row["city_name"])
    province_to_region[row["province_name"]].add(row["region_name"])

In [42]:
# proper search lists
ALL_BARANGAYS = dim["barangay_name"].dropna().unique().tolist()
ALL_CITIES = dim["city_name"].dropna().unique().tolist()
ALL_PROVINCES = dim["province_name"].dropna().unique().tolist()

# Manila appears as district-level "city_name" entries in dim. Keep a normalized set for fallback detection.
MANILA_DISTRICTS = {
    "BINONDO", "ERMITA", "INTRAMUROS", "MALATE", "PACO", "PANDACAN",
    "PORT AREA", "QUIAPO", "SAMPALOC", "SAN ANDRES", "SAN MIGUEL",
    "SAN NICOLAS", "SANTA ANA", "SANTA CRUZ", "TONDO I/II", "TONDO I II"
}

# Restrict to Manila district tokens that are actually present in dim city_name values.
MANILA_DISTRICTS_IN_DIM = [
    c for c in ALL_CITIES
    if c in MANILA_DISTRICTS or c.replace("/", " ") in MANILA_DISTRICTS
]

# Canonical province string used by dim for Manila records.
MANILA_PROVINCE = "METRO MANILA"

# Candidate Extraction (fuzzy-enabled)

In [43]:
from rapidfuzz import fuzz

def detect_province(addr_clean, threshold=85):
    matches = []

    for prov in ALL_PROVINCES:
        score = fuzz.partial_ratio(addr_clean, prov)
        if score >= threshold:
            matches.append((prov, score))

    return sorted(matches, key=lambda x: x[1], reverse=True)

In [44]:
def get_cities_in_province(province):
    return dim.loc[dim["province_name"] == province, "city_name"].unique().tolist()

In [45]:
def detect_city(addr_clean, candidate_cities, threshold=85):
    matches = []

    for city in candidate_cities:
        score = fuzz.partial_ratio(addr_clean, city)
        if score >= threshold:
            matches.append((city, score))

    return sorted(matches, key=lambda x: x[1], reverse=True)

In [46]:
def get_barangays_in_city(city):
    return dim.loc[dim["city_name"] == city, "barangay_name"].unique().tolist()

In [47]:
def detect_barangay(addr_clean, candidate_barangays, threshold=85):
    matches = []

    for bgy in candidate_barangays:
        score = fuzz.partial_ratio(addr_clean, bgy)
        if score >= threshold:
            matches.append((bgy, score))

    return sorted(matches, key=lambda x: x[1], reverse=True)

# Core Parsing Logic


In [48]:
def parse_row_hierarchical(addr_clean):

    # 1. PROVINCE
    prov_matches = detect_province(addr_clean)

    # Fallback: handle Manila addresses written as districts (e.g., BINONDO, ERMITA, TONDO I/II).
    if not prov_matches:
        manila_city_matches = detect_city(addr_clean, MANILA_DISTRICTS_IN_DIM, threshold=80)
        if manila_city_matches:
            best_city, city_score = manila_city_matches[0]
            best_prov = MANILA_PROVINCE
            prov_score = fuzz.partial_ratio(addr_clean, MANILA_PROVINCE)
        else:
            return pd.Series([None, None, None, None, 0])
    else:
        best_prov, prov_score = prov_matches[0]

        # 2. CITY (within province)
        cities = get_cities_in_province(best_prov)
        city_matches = detect_city(addr_clean, cities)

        if city_matches:
            best_city, city_score = city_matches[0]
        else:
            best_city, city_score = None, 0

    # 3. BARANGAY (within city)
    if best_city:
        barangays = get_barangays_in_city(best_city)
        bgy_matches = detect_barangay(addr_clean, barangays)

        if bgy_matches:
            best_bgy, bgy_score = bgy_matches[0]
        else:
            best_bgy, bgy_score = None, 0
    else:
        best_bgy, bgy_score = None, 0

    # 4. REGION
    region = None
    if best_prov:
        region_list = list(province_to_region.get(best_prov, []))
        region = region_list[0] if region_list else None

    # 5. FINAL SCORE
    final_score = (
        0.5 * bgy_score +
        0.3 * city_score +
        0.2 * prov_score
    )

    return pd.Series([
        best_bgy,
        best_city,
        best_prov,
        region,
        final_score
    ])

# Apply to df

In [49]:
addr[["barangay", "city", "province", "region", "score"]] = (
    addr["clean_address"].apply(parse_row_hierarchical)
)

In [50]:
addr.to_csv("../../data/output/valid/pipeline_test.csv")